<a href="https://colab.research.google.com/github/naman39910/voyage-analytics/blob/main/nootbooks/regression_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VOYAGE ANALYTICS — FLIGHT PRICE REGRESSION MODEL

### Goal : Predict flight price using flights.csv
### Features: distance, flightType, agency, month, day_of_week
###Target : price

### IMPORTS


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


### LOAD DATA

In [2]:
# Path relative to notebooks folder
DATA_PATH = '/content/flights.csv'

flights = pd.read_csv(DATA_PATH)

print("=" * 50)
print("FLIGHTS DATASET LOADED")
print("=" * 50)
print(f"Shape          : {flights.shape}")
print(f"Rows           : {flights.shape[0]:,}")
print(f"Columns        : {flights.shape[1]}")
print(f"\nColumn Names   : {flights.columns.tolist()}")
print(f"\nData Types:\n{flights.dtypes}")
print(f"\nFirst 5 Rows:")
flights.head()

FLIGHTS DATASET LOADED
Shape          : (271888, 10)
Rows           : 271,888
Columns        : 10

Column Names   : ['travelCode', 'userCode', 'from', 'to', 'flightType', 'price', 'time', 'distance', 'agency', 'date']

Data Types:
travelCode      int64
userCode        int64
from           object
to             object
flightType     object
price         float64
time          float64
distance      float64
agency         object
date           object
dtype: object

First 5 Rows:


,travelCode,userCode,from,to,flightType,price,time,distance,agency,date
0,0,0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,09/26/2019
1,0,0,Florianopolis (SC),Recife (PE),firstClass,1292.29,1.76,676.53,FlyingDrops,09/30/2019
2,1,0,Brasilia (DF),Florianopolis (SC),firstClass,1487.52,1.66,637.56,CloudFy,10/03/2019
3,1,0,Florianopolis (SC),Brasilia (DF),firstClass,1127.36,1.66,637.56,CloudFy,10/04/2019
4,2,0,Aracaju (SE),Salvador (BH),firstClass,1684.05,2.16,830.86,CloudFy,10/10/2019


### QUICK DATA VALIDATION

In [3]:
print("MISSING VALUES:")
print(flights.isnull().sum())

print(f"\nDUPLICATE ROWS: {flights.duplicated().sum()}")

print(f"\nTARGET VARIABLE (price) STATS:")
print(flights['price'].describe())

print(f"\nFLIGHT TYPE DISTRIBUTION:")
print(flights['flightType'].value_counts())

print(f"\nAGENCY DISTRIBUTION:")
print(flights['agency'].value_counts())

MISSING VALUES:
travelCode    0
userCode      0
from          0
to            0
flightType    0
price         0
time          0
distance      0
agency        0
date          0
dtype: int64

DUPLICATE ROWS: 0

TARGET VARIABLE (price) STATS:
count    271888.00000
mean        957.37503
std         362.31189
min         301.51000
25%         672.66000
50%         904.00000
75%        1222.24000
max        1754.17000
Name: price, dtype: float64

FLIGHT TYPE DISTRIBUTION:
flightType
firstClass    116418
premium        78004
economic       77466
Name: count, dtype: int64

AGENCY DISTRIBUTION:
agency
Rainbow        116752
CloudFy        116378
FlyingDrops     38758
Name: count, dtype: int64


### FEATURE ENGINEERING

In [4]:
# EDA Insights Applied:
# Drop 'time' → perfectly correlated with distance (r=1.000)
# Extract month from date → seasonal patterns found
# Drop travelCode, userCode → just IDs
# Drop from, to → too many unique route combinations
# ============================================================

def engineer_features(df):
    """Feature engineering pipeline based on EDA insights."""

    df = df.copy()

    # Parse date
    df['date'] = pd.to_datetime(df['date'])

    # Extract time features
    df['month']       = df['date'].dt.month
    df['day_of_week'] = df['date'].dt.dayofweek

    # Drop columns
    cols_to_drop = [
        'travelCode',   # ID only
        'userCode',     # ID only
        'date',         # replaced by month + day_of_week
        'time',         # perfectly correlated with distance (r=1.000)
        'from',         # too many categories
        'to',           # too many categories
    ]
    df = df.drop(columns=cols_to_drop, errors='ignore')

    return df


flights_featured = engineer_features(flights)

print("AFTER FEATURE ENGINEERING:")
print(f"Shape   : {flights_featured.shape}")
print(f"Columns : {flights_featured.columns.tolist()}")
print(f"\nSample:")
flights_featured.head()

AFTER FEATURE ENGINEERING:
Shape   : (271888, 6)
Columns : ['flightType', 'price', 'distance', 'agency', 'month', 'day_of_week']

Sample:


,flightType,price,distance,agency,month,day_of_week
0,firstClass,1434.38,676.53,FlyingDrops,9,3
1,firstClass,1292.29,676.53,FlyingDrops,9,0
2,firstClass,1487.52,637.56,CloudFy,10,3
3,firstClass,1127.36,637.56,CloudFy,10,4
4,firstClass,1684.05,830.86,CloudFy,10,3


### ENCODE CATEGORICAL VARIABLES


In [5]:
def encode_features(df):
    """
    Encode categorical columns.

    flightType -> Ordinal (economic=0, premium=1, firstClass=2)
                 Natural price ordering confirmed in EDA
    agency     -> One-Hot (no natural order between agencies)
    """
    df = df.copy()

    # Ordinal encoding for flightType
    flight_type_map = {
        'economic': 0, 'premium': 1, 'firstClass': 2
    }
    df['flightType_encoded'] = df['flightType'].map(flight_type_map)

    # One-hot encoding for agency
    agency_dummies = pd.get_dummies(
        df['agency'], prefix='agency', drop_first=False   # keep all 3 for interpretability
    )
    df = pd.concat([df, agency_dummies], axis=1)

    # Drop original categorical columns
    df = df.drop(columns=['flightType', 'agency'])

    return df


flights_encoded = encode_features(flights_featured)

print("AFTER ENCODING:")
print(f"Shape   : {flights_encoded.shape}")
print(f"Columns : {flights_encoded.columns.tolist()}")
print(f"\nSample:")
flights_encoded.head()

AFTER ENCODING:
Shape   : (271888, 8)
Columns : ['price', 'distance', 'month', 'day_of_week', 'flightType_encoded', 'agency_CloudFy', 'agency_FlyingDrops', 'agency_Rainbow']

Sample:


,price,distance,month,day_of_week,flightType_encoded,agency_CloudFy,agency_FlyingDrops,agency_Rainbow
0,1434.38,676.53,9,3,2,False,True,False
1,1292.29,676.53,9,0,2,False,True,False
2,1487.52,637.56,10,3,2,True,False,False
3,1127.36,637.56,10,4,2,True,False,False
4,1684.05,830.86,10,3,2,True,False,False


### PREPARE FEATURES (X) AND TARGET (y)


In [6]:
TARGET = 'price'

X = flights_encoded.drop(columns=[TARGET])
y = flights_encoded[TARGET]

print(f"Feature Matrix X : {X.shape}")
print(f"Target Vector  y : {y.shape}")
print(f"\nFeature Columns  : {X.columns.tolist()}")
print(f"\nTarget Stats:")
print(y.describe())
print(f"\nMissing in X : {X.isnull().sum().sum()}")
print(f"Missing in y : {y.isnull().sum()}")

Feature Matrix X : (271888, 7)
Target Vector  y : (271888,)

Feature Columns  : ['distance', 'month', 'day_of_week', 'flightType_encoded', 'agency_CloudFy', 'agency_FlyingDrops', 'agency_Rainbow']

Target Stats:
count    271888.00000
mean        957.37503
std         362.31189
min         301.51000
25%         672.66000
50%         904.00000
75%        1222.24000
max        1754.17000
Name: price, dtype: float64

Missing in X : 0
Missing in y : 0


### TRAIN-TEST SPLIT

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 80% train / 20% test
    random_state=42     # reproducibility
)

print(f"Total Samples  : {len(X):,}")
print(f"Training Set   : {len(X_train):,} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test Set       : {len(X_test):,}  ({len(X_test)/len(X)*100:.0f}%)")

# Scale features (needed for Linear Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"\n-> Train-Test Split Complete")
print(f"-> StandardScaler Fitted")

Total Samples  : 271,888
Training Set   : 217,510 (80%)
Test Set       : 54,378  (20%)

-> Train-Test Split Complete
-> StandardScaler Fitted


In [8]:
print(f"Missing values in X_train: {X_train.isnull().sum().sum()}")

Missing values in X_train: 0


### TRAIN AND COMPARE MODELS


In [9]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Train model and compute evaluation metrics."""

    model.fit(X_tr, y_tr)

    y_pred_train = model.predict(X_tr)
    y_pred_test  = model.predict(X_te)

    metrics = {
        'Model'    : name,
        'Train R²' : round(r2_score(y_tr, y_pred_train), 4),
        'Test R²'  : round(r2_score(y_te, y_pred_test),  4),
        'MAE'      : round(mean_absolute_error(y_te, y_pred_test), 2),
        'RMSE'     : round(np.sqrt(mean_squared_error(y_te, y_pred_test)), 2),
        'MAPE %'   : round(np.mean(np.abs((y_te - y_pred_test) / y_te)) * 100, 2)
    }

    return metrics, model


# Define models
models_to_try = {
    'Linear Regression' : LinearRegression(),

    'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),

    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
}
# Train all models
results       = []
trained_models = {}

print("TRAINING MODELS...")
print("=" * 60)

for name, model in models_to_try.items():
    print(f"  Training {name}...", end='')

    # Linear Regression uses scaled data
    # Tree models use raw data
    if name == 'Linear Regression':
        metrics, trained = evaluate_model(
            name, model,
            X_train_scaled, X_test_scaled,
            y_train, y_test
        )
    else:
        metrics, trained = evaluate_model(
            name, model,
            X_train, X_test,
            y_train, y_test
        )

    results.append(metrics)
    trained_models[name] = trained
    print(f" ->  Test R²: {metrics['Test R²']} | MAE: ${metrics['MAE']}")

# Results table
print("\n")
results_df = pd.DataFrame(results).sort_values('Test R²', ascending=False)
print("=" * 70)
print("MODEL COMPARISON RESULTS")
print("=" * 70)
print(results_df.to_string(index=False))

TRAINING MODELS...
  Training Linear Regression... ->  Test R²: 0.7703 | MAE: $134.58
  Training Random Forest... ->  Test R²: 0.9067 | MAE: $61.8
  Training Gradient Boosting... ->  Test R²: 0.8819 | MAE: $86.34


MODEL COMPARISON RESULTS
            Model  Train R²  Test R²    MAE   RMSE  MAPE %
    Random Forest    0.9167   0.9067  61.80 110.90    7.08
Gradient Boosting    0.8830   0.8819  86.34 124.73    9.81
Linear Regression    0.7713   0.7703 134.58 173.98   16.59


### SAVE MODEL ARTIFACTS

In [10]:
ARTIFACTS_DIR = 'model_artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Save the StandardScaler
scaler_filename = os.path.join(ARTIFACTS_DIR, 'scaler.joblib')
joblib.dump(scaler, scaler_filename)
print(f"Scaler saved to {scaler_filename}")

# Save each trained model
for name, model in trained_models.items():
    model_filename = os.path.join(ARTIFACTS_DIR, f'{name.replace(" ", "_").lower()}_model.joblib')
    joblib.dump(model, model_filename)
    print(f"Model '{name}' saved to {model_filename}")

print("\nAll model artifacts saved successfully!")

Scaler saved to model_artifacts/scaler.joblib
Model 'Linear Regression' saved to model_artifacts/linear_regression_model.joblib
Model 'Random Forest' saved to model_artifacts/random_forest_model.joblib
Model 'Gradient Boosting' saved to model_artifacts/gradient_boosting_model.joblib

All model artifacts saved successfully!


### Create Project Directory Structure

In [11]:
import os

project_root = 'travel-mlops-project'

# Create main project directory
os.makedirs(project_root, exist_ok=True)

# Create subdirectories
os.makedirs(os.path.join(project_root, 'model'), exist_ok=True)
os.makedirs(os.path.join(project_root, 'api'), exist_ok=True)
os.makedirs(os.path.join(project_root, 'notebooks'), exist_ok=True)
os.makedirs(os.path.join(project_root, 'data'), exist_ok=True)

# Create placeholder files
open(os.path.join(project_root, 'model', 'flight_price_model.pkl'), 'a').close()
open(os.path.join(project_root, 'api', 'app.py'), 'a').close()
open(os.path.join(project_root, 'requirements.txt'), 'a').close()

print(f"Project directory structure '{project_root}/' created successfully!")

# Verify creation (optional)
print("\nVerifying structure:")
!ls -R {project_root}

Project directory structure 'travel-mlops-project/' created successfully!

Verifying structure:
travel-mlops-project:
api  data  model  notebooks  requirements.txt

travel-mlops-project/api:
app.py

travel-mlops-project/data:

travel-mlops-project/model:
flight_price_model.pkl

travel-mlops-project/notebooks:


In [12]:
import os

os.listdir()

['.config',
 'drive',
 'flights.csv',
 'model_artifacts',
 'travel-mlops-project',
 'sample_data']

In [13]:
os.listdir("model_artifacts")

['random_forest_model.joblib',
 'linear_regression_model.joblib',
 'scaler.joblib',
 'gradient_boosting_model.joblib']

In [14]:
from google.colab import files

files.download("model_artifacts/random_forest_model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
files.download("model_artifacts/scaler.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
print(X.shape)


(271888, 7)


In [17]:
print(X.columns)

Index(['distance', 'month', 'day_of_week', 'flightType_encoded',
       'agency_CloudFy', 'agency_FlyingDrops', 'agency_Rainbow'],
      dtype='object')
